# Yelp — EmbeddingANN Optimised (Optuna)
Optuna TPE search over all ANN hyperparameters including architecture depth (1-5 layers).
**Saves to**: `optimization_ANN/`


## Roadmap
```
STEP 1  — Imports & paths
STEP 2  — Load data & encoders
STEP 3  — Encode categoricals
STEP 4  — Scale numerical features
STEP 5  — Dataset & DataLoader factory
STEP 6  — Flexible model definition
STEP 7  — Optuna HPO (50 trials)
STEP 8  — Retrain final model
STEP 9  — Evaluate
STEP 10 — Save optimised embeddings
```

---
## Step 1 — Imports & Paths

In [ ]:
import os, json, time, warnings, shutil
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import optuna
from optuna.pruners  import MedianPruner
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED   = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Device: {DEVICE}  Optuna: {optuna.__version__}")


In [ ]:
DATA_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\data"
EMB_DIR  = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\embeddings"
OUT_DIR  = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\optimization_ANN"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Out dir: {OUT_DIR}")


---
## Step 2 — Load Data & Encoders

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train_features.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val_features.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test_features.csv'))
print(f"Train : {df_train.shape}  Val : {df_val.shape}  Test : {df_test.shape}")


In [ ]:
NUM_COLS  = [
    'user_avg_rating',    'user_rating_count',
    'business_avg_rating','business_rating_count', 'business_rating_std',
    'days_since_review'
]
TARGET = 'stars'


In [ ]:
with open(os.path.join(EMB_DIR,'state_encoder.json'))    as f: state2idx = json.load(f)
with open(os.path.join(EMB_DIR,'city_encoder.json'))     as f: city2idx  = json.load(f)
with open(os.path.join(EMB_DIR,'cat_encoder.json'))      as f: cat2idx   = json.load(f)
with open(os.path.join(EMB_DIR,'user_encoder.json'))     as f: user2idx  = json.load(f)
with open(os.path.join(EMB_DIR,'business_encoder.json')) as f: biz2idx   = json.load(f)

n_states = len(state2idx); n_cities = len(city2idx)
n_cats   = len(cat2idx);   n_users  = len(user2idx); n_biz = len(biz2idx)
print(f"State:{n_states} City:{n_cities} Cat:{n_cats} User:{n_users:,} Biz:{n_biz:,}")


---
## Step 3 — Encode Categoricals

In [ ]:
MAX_CATS = 10

def encode_split(df):
    state_idx = df['state'].fillna('<UNK>').map(lambda x: state2idx.get(x,0)).values.astype(np.int64)
    city_idx  = df['city'].fillna('<UNK>').map(lambda x: city2idx.get(x,0)).values.astype(np.int64)
    cat_seqs = []
    for c in df['categories'].fillna('<UNK>'):
        tokens  = c.split('|')[:MAX_CATS]
        indices = [cat2idx.get(t,0) for t in tokens]
        indices += [-1]*(MAX_CATS-len(indices))
        cat_seqs.append(indices)
    cat_idx  = np.array(cat_seqs, dtype=np.int64)
    user_idx = df['user_id'].map(lambda x: user2idx.get(x,0)).values.astype(np.int64)
    biz_idx  = df['business_id'].map(lambda x: biz2idx.get(x,0)).values.astype(np.int64)
    return state_idx, city_idx, cat_idx, user_idx, biz_idx

tr_state,tr_city,tr_cats,tr_user,tr_biz = encode_split(df_train)
v_state, v_city, v_cats, v_user, v_biz  = encode_split(df_val)
te_state,te_city,te_cats,te_user,te_biz = encode_split(df_test)
print("Encoding done ✓")


---
## Step 4 — Scale Numerical Features

In [ ]:
scaler      = StandardScaler()
X_train_num = scaler.fit_transform(df_train[NUM_COLS]).astype(np.float32)
X_val_num   = scaler.transform(df_val[NUM_COLS]).astype(np.float32)
X_test_num  = scaler.transform(df_test[NUM_COLS]).astype(np.float32)
y_train = df_train[TARGET].values.astype(np.float32)
y_val   = df_val[TARGET].values.astype(np.float32)
y_test  = df_test[TARGET].values.astype(np.float32)
print(f"Numerical matrix: {X_train_num.shape}")


---
## Step 5 — Dataset & DataLoader Factory

In [ ]:
class YelpDataset(Dataset):
    def __init__(self, num, state, city, cats, user, biz, labels):
        self.num   = torch.tensor(num,   dtype=torch.float32)
        self.state = torch.tensor(state, dtype=torch.long)
        self.city  = torch.tensor(city,  dtype=torch.long)
        self.cats  = torch.tensor(cats,  dtype=torch.long)
        self.user  = torch.tensor(user,  dtype=torch.long)
        self.biz   = torch.tensor(biz,   dtype=torch.long)
        self.y     = torch.tensor(labels,dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return (self.num[idx],self.state[idx],self.city[idx],
                self.cats[idx],self.user[idx],self.biz[idx],self.y[idx])

def make_loaders(batch_size):
    tr = DataLoader(YelpDataset(X_train_num,tr_state,tr_city,tr_cats,tr_user,tr_biz,y_train),
                    batch_size=batch_size,shuffle=True)
    v  = DataLoader(YelpDataset(X_val_num,  v_state, v_city, v_cats, v_user, v_biz, y_val),
                    batch_size=batch_size,shuffle=False)
    te = DataLoader(YelpDataset(X_test_num, te_state,te_city,te_cats,te_user,te_biz,y_test),
                    batch_size=batch_size,shuffle=False)
    return tr, v, te


---
## Step 6 — Flexible Model Definition (n_layers: 1–5)

In [ ]:
class FlexYelpANN(nn.Module):
    def __init__(self, n_states, n_cities, n_cats, n_users, n_biz, n_num,
                 state_dim, city_dim, cat_dim, user_dim, biz_dim,
                 n_layers, hidden_dim, emb_dropout, mlp_dropout):
        super().__init__()
        self.state_emb = nn.Embedding(n_states, state_dim, padding_idx=0)
        self.city_emb  = nn.Embedding(n_cities, city_dim,  padding_idx=0)
        self.cat_emb   = nn.Embedding(n_cats,   cat_dim,   padding_idx=0)
        self.user_emb  = nn.Embedding(n_users,  user_dim,  padding_idx=0)
        self.biz_emb   = nn.Embedding(n_biz,    biz_dim,   padding_idx=0)
        self.emb_drop  = nn.Dropout(emb_dropout)
        self.n_states=n_states; self.n_cities=n_cities
        self.n_cats=n_cats;     self.n_users=n_users; self.n_biz=n_biz

        mlp_in = state_dim+city_dim+cat_dim+user_dim+biz_dim+n_num
        layers = []
        in_dim = mlp_in
        for _ in range(n_layers):
            layers += [nn.Linear(in_dim,hidden_dim),nn.ReLU(),nn.Dropout(mlp_dropout)]
            in_dim  = hidden_dim
        layers += [nn.Linear(in_dim,1)]
        self.mlp = nn.Sequential(*layers)

    def forward(self, num, state, city, cats, user, biz):
        state = state.clamp(0,self.n_states-1); city = city.clamp(0,self.n_cities-1)
        user  = user.clamp(0, self.n_users-1);  biz  = biz.clamp(0, self.n_biz-1)
        mask     = (cats>=0).float()
        safe_cats= cats.clamp(0,self.n_cats-1)
        cat_vecs = self.cat_emb(safe_cats)*mask.unsqueeze(-1)
        cat_vec  = cat_vecs.sum(1)/mask.sum(1).clamp(min=1).unsqueeze(-1)
        sv = self.state_emb(state); cv = self.city_emb(city)
        uv = self.emb_drop(self.user_emb(user)); bv = self.emb_drop(self.biz_emb(biz))
        return self.mlp(torch.cat([sv,cv,cat_vec,uv,bv,num],dim=1)).squeeze(1)


---
## Step 7 — Optuna HPO (50 trials)

Tunes embedding dims per feature, architecture depth (1-5 layers),
hidden width, dropout rates, learning rate, weight decay, and batch size.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    model.train() if optimizer else model.eval()
    preds_all, labels_all = [], []
    ctx = torch.enable_grad() if optimizer else torch.no_grad()
    with ctx:
        for num_b,st_b,ci_b,ca_b,us_b,bz_b,y_b in loader:
            num_b,st_b,ci_b,ca_b,us_b,bz_b,y_b = (
                num_b.to(DEVICE),st_b.to(DEVICE),ci_b.to(DEVICE),
                ca_b.to(DEVICE), us_b.to(DEVICE),bz_b.to(DEVICE),y_b.to(DEVICE))
            preds = model(num_b,st_b,ci_b,ca_b,us_b,bz_b)
            loss  = criterion(preds,y_b)
            if optimizer:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            preds_all.extend(preds.detach().cpu().numpy())
            labels_all.extend(y_b.cpu().numpy())
    p=np.array(preds_all); l=np.array(labels_all)
    return round(float(np.sqrt(mean_squared_error(l,p))),4)


In [ ]:
N_EPOCHS_TRIAL = 10; PATIENCE_TRIAL = 3

def objective(trial):
    p = {
        'state_dim'  : trial.suggest_categorical('state_dim',  [2,4,8]),
        'city_dim'   : trial.suggest_categorical('city_dim',   [4,8,16]),
        'cat_dim'    : trial.suggest_categorical('cat_dim',    [4,8,16]),
        'user_dim'   : trial.suggest_categorical('user_dim',   [8,16,32]),
        'biz_dim'    : trial.suggest_categorical('biz_dim',    [8,16,32]),
        'n_layers'   : trial.suggest_int('n_layers',   1, 5),
        'hidden_dim' : trial.suggest_int('hidden_dim', 32, 256, step=32),
        'emb_dropout': trial.suggest_float('emb_dropout', 0.1, 0.5, step=0.1),
        'mlp_dropout': trial.suggest_float('mlp_dropout', 0.1, 0.4, step=0.1),
        'lr'         : trial.suggest_float('lr',          1e-4, 1e-2, log=True),
        'weight_decay': trial.suggest_float('weight_decay',1e-6, 1e-3, log=True),
        'batch_size' : trial.suggest_categorical('batch_size', [256,512,1024]),
    }
    tr_l,v_l,_ = make_loaders(p['batch_size'])
    m = FlexYelpANN(n_states,n_cities,n_cats,n_users,n_biz,len(NUM_COLS),
                    p['state_dim'],p['city_dim'],p['cat_dim'],
                    p['user_dim'],p['biz_dim'],p['n_layers'],
                    p['hidden_dim'],p['emb_dropout'],p['mlp_dropout']).to(DEVICE)
    opt  = optim.Adam(m.parameters(),lr=p['lr'],weight_decay=p['weight_decay'])
    crit = nn.MSELoss()
    best_val=float('inf'); patience=0
    for epoch in range(N_EPOCHS_TRIAL):
        run_epoch(m,tr_l,crit,opt)
        val_rmse = run_epoch(m,v_l,crit)
        trial.report(val_rmse,epoch)
        if trial.should_prune(): raise optuna.exceptions.TrialPruned()
        if val_rmse < best_val: best_val=val_rmse; patience=0
        else:
            patience+=1
            if patience>=PATIENCE_TRIAL: break
    return best_val

study = optuna.create_study(direction='minimize',
    sampler=TPESampler(seed=SEED), pruner=MedianPruner(n_warmup_steps=5))
print("Starting Optuna — 50 trials...")
t0 = time.perf_counter()
study.optimize(objective, n_trials=50, show_progress_bar=True)
print(f"HPO done in {(time.perf_counter()-t0)/60:.1f} min  Best: {study.best_value:.4f}")
for k,v in study.best_params.items(): print(f"  {k:<20} = {v}")


---
## Step 8 — Retrain Final Model

In [ ]:
p = study.best_params
N_EPOCHS_FINAL=30; PATIENCE_FINAL=5
tr_l,v_l,te_l = make_loaders(p['batch_size'])

final_model = FlexYelpANN(n_states,n_cities,n_cats,n_users,n_biz,len(NUM_COLS),
    p['state_dim'],p['city_dim'],p['cat_dim'],p['user_dim'],p['biz_dim'],
    p['n_layers'],p['hidden_dim'],p['emb_dropout'],p['mlp_dropout']).to(DEVICE)

optimizer = optim.Adam(final_model.parameters(),lr=p['lr'],weight_decay=p['weight_decay'])
criterion = nn.MSELoss()
best_val=float('inf'); best_epoch=0; patience_cnt=0; best_state=None

t0 = time.perf_counter()
for epoch in range(N_EPOCHS_FINAL):
    run_epoch(final_model,tr_l,criterion,optimizer)
    val_rmse = run_epoch(final_model,v_l,criterion)
    if val_rmse < best_val:
        best_val=val_rmse; best_epoch=epoch+1; patience_cnt=0
        best_state={k:v.clone() for k,v in final_model.state_dict().items()}
    else:
        patience_cnt+=1
    if (epoch+1)%5==0 or epoch==0:
        print(f"Epoch {epoch+1:>2}  Val:{val_rmse}  "
              f"{'best' if patience_cnt==0 else f'pat {patience_cnt}/{PATIENCE_FINAL}'}")
    if patience_cnt>=PATIENCE_FINAL:
        print(f"Early stopping at epoch {epoch+1}"); break

final_model.load_state_dict(best_state)
train_time = time.perf_counter()-t0
print(f"Best val RMSE: {best_val} at epoch {best_epoch}  Time: {train_time:.1f}s")


---
## Step 9 — Evaluate

In [ ]:
def full_metrics(model, loader):
    model.eval(); preds_all,labels_all=[],[]
    with torch.no_grad():
        for num_b,st_b,ci_b,ca_b,us_b,bz_b,y_b in loader:
            num_b,st_b,ci_b,ca_b,us_b,bz_b = (num_b.to(DEVICE),st_b.to(DEVICE),
                ci_b.to(DEVICE),ca_b.to(DEVICE),us_b.to(DEVICE),bz_b.to(DEVICE))
            preds_all.extend(final_model(num_b,st_b,ci_b,ca_b,us_b,bz_b).cpu().numpy())
            labels_all.extend(y_b.numpy())
    p=np.array(preds_all); l=np.array(labels_all)
    return (round(float(np.sqrt(mean_squared_error(l,p))),4),
            round(float(mean_absolute_error(l,p)),4))

tr_rmse,tr_mae   = full_metrics(final_model,tr_l)
val_rmse,val_mae = full_metrics(final_model,v_l)
te_rmse,te_mae   = full_metrics(final_model,te_l)

print("="*55)
print("YelpEmbeddingANN (Optimised) — Final Results")
print("="*55)
print(f"  Train      RMSE: {tr_rmse}   MAE: {tr_mae}")
print(f"  Validation RMSE: {val_rmse}   MAE: {val_mae}")
print(f"  Test       RMSE: {te_rmse}   MAE: {te_mae}")
print(f"  Train time : {train_time:.1f}s  Best epoch: {best_epoch}")
print(f"  Train-Test gap : {round(te_rmse-tr_rmse,4)}")


---
## Step 10 — Save Optimised Embeddings

In [ ]:
weight_map = {
    'state_embeddings'   : final_model.state_emb,
    'city_embeddings'    : final_model.city_emb,
    'cat_embeddings'     : final_model.cat_emb,
    'user_embeddings'    : final_model.user_emb,
    'business_embeddings': final_model.biz_emb,
}
for fname, layer in weight_map.items():
    w = layer.weight.detach().cpu().numpy()
    np.save(os.path.join(OUT_DIR, f'{fname}.npy'), w)
    print(f"  {fname}.npy  {w.shape}")

for enc in ['state_encoder.json','city_encoder.json','cat_encoder.json',
            'user_encoder.json','business_encoder.json']:
    shutil.copy(os.path.join(EMB_DIR,enc), os.path.join(OUT_DIR,enc))

result = {'model':'YelpEmbeddingANN (Optimised)',
    'train_rmse':tr_rmse,'val_rmse':val_rmse,'test_rmse':te_rmse,
    'train_mae':tr_mae,  'val_mae':val_mae,  'test_mae':te_mae,
    'train_time_s':round(train_time,1),'best_epoch':best_epoch,
    'best_params':study.best_params}
with open(os.path.join(OUT_DIR,'ann_results.json'),'w') as f:
    json.dump(result,f,indent=2)
print(f"All files saved → {OUT_DIR}")
